# Meet your cluster

Before any of the clever parts, one question: how fast can these GPUs actually talk to
each other?

It matters more than it sounds like it should. Two GPUs in the same chassis might be
joined by a wire that moves 270 GB/s, or by one that moves 23. Same box, same code,
more than ten times the difference. Every decision later in this course comes back to
that number.

![Not every link runs at the same speed](../assets/diagrams/interconnect.png)

This is why tensor parallelism stays inside a node and data parallelism is allowed to
cross between them. You put the chatty collective on the fast wire and the occasional
one on the slow wire.

So we measure the machine first, before any theory.

**In this notebook:** report the environment, read the topology matrix, measure
peer-to-peer and host bandwidth, and work out how much math this machine can do in the
time it takes to send a gigabyte.

---
## What hardware is this

Start with what the driver reports.

In [ ]:
import torch

print(f"torch          : {torch.__version__}")
print(f"cuda runtime   : {torch.version.cuda}")
print(f"nccl available : {torch.distributed.is_nccl_available()}")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()}")

NUM_GPUS = torch.cuda.device_count()
print(f"gpus           : {NUM_GPUS}\n")

for i in range(NUM_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"  gpu {i}: {p.name}  {p.total_memory / 1024**3:.0f} GB  sm_{p.major}{p.minor}  {p.multi_processor_count} SMs")

---
## Which wire connects which pair

`nvidia-smi topo -m` prints the link type between every pair of devices. The codes,
fastest first:

| Code | Meaning | Rough bandwidth |
|---|---|---|
| `NV#` | NVLink, `#` is the number of links | 25 GB/s per link |
| `PIX` | Same PCIe switch | ~25 GB/s |
| `PXB` | Across PCIe switches | ~15 GB/s |
| `PHB` | Through the PCIe host bridge | ~12 GB/s |
| `NODE` | Across host bridges, same NUMA node | ~10 GB/s |
| `SYS` | Across NUMA nodes, over the CPU interconnect | slowest |

`NV12` means twelve NVLink connections between that pair, which is the best case: every
GPU reaches every other at full speed, and no pair has to fall back to PCIe.

In [ ]:
import subprocess

topo = subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True)
print(topo.stdout)

---
## Measure it, do not trust it

Specification bandwidth is a marketing number. Copy real tensors and time them.

We copy a large buffer from one GPU to another, repeatedly, and divide bytes by
seconds. `torch.cuda.synchronize()` matters here: CUDA calls are asynchronous, so
without it you would be timing how fast Python can queue work, not how fast the
wire moves data. It also has to name the devices: a bare `synchronize()` waits
only on the *current* device, which silently reports impossible numbers for
any pair that does not involve it.

In [ ]:
import time
import torch

BUFFER_BYTES = 256 * 1024 * 1024   # 256 MB: large enough to leave launch overhead behind
WARMUP_ITERS = 3
TIMED_ITERS = 10


def sync(*devices):
    """Wait for real completion.

    A bare torch.cuda.synchronize() only waits on the CURRENT device. Copying
    between gpu2 and gpu3 while device 0 is current would return instantly and
    you would time how fast Python queues work, not how fast the wire moves.
    """
    for d in devices:
        if d != "cpu":
            torch.cuda.synchronize(d)


def copy_bandwidth_gbps(src_device, dst_device, n_bytes=BUFFER_BYTES):
    """Sustained copy bandwidth in GB/s. Devices may be ints (gpu) or 'cpu'."""
    src = torch.empty(n_bytes // 4, dtype=torch.float32, device=src_device)
    dst = torch.empty_like(src, device=dst_device)

    for _ in range(WARMUP_ITERS):
        dst.copy_(src)
    sync(src_device, dst_device)

    start = time.perf_counter()
    for _ in range(TIMED_ITERS):
        dst.copy_(src)
    sync(src_device, dst_device)
    elapsed = time.perf_counter() - start

    return (n_bytes * TIMED_ITERS) / elapsed / 1e9


# GPU-to-GPU matrix. The diagonal is intra-device, so we skip it.
print("peer-to-peer copy bandwidth (GB/s)\n")
print("      " + "".join(f"{f'gpu{j}':>9}" for j in range(NUM_GPUS)))
for i in range(NUM_GPUS):
    row = f"gpu{i}  "
    for j in range(NUM_GPUS):
        row += "        -" if i == j else f"{copy_bandwidth_gbps(i, j):>9.1f}"
    print(row)

---
## The slow wire, for contrast

Now the same measurement across PCIe to host memory. This is the number that makes
CPU offloading (ZeRO-Offload, module 05) expensive, and it is why you never casually
move a tensor to the CPU inside a training loop.

In [ ]:
h2d = copy_bandwidth_gbps("cpu", 0)
d2h = copy_bandwidth_gbps(0, "cpu")
p2p = copy_bandwidth_gbps(0, 1) if NUM_GPUS > 1 else float("nan")

print(f"host -> gpu0   : {h2d:7.1f} GB/s")
print(f"gpu0 -> host   : {d2h:7.1f} GB/s")
print(f"gpu0 -> gpu1   : {p2p:7.1f} GB/s")
print(f"\nGPU-to-GPU is {p2p / h2d:.1f}x faster than GPU-to-host on this machine.")

---
## The ratio that decides everything

A GPU's compute rate and its communication rate set a budget. If a layer does
little math but moves a lot of data, you are network-bound and no amount of
parallelism will save you.

Measure raw matmul throughput, then compare.

In [ ]:
MATMUL_DIM = 8192
MATMUL_ITERS = 50


def matmul_tflops(dtype):
    a = torch.randn(MATMUL_DIM, MATMUL_DIM, device=0, dtype=dtype)
    b = torch.randn(MATMUL_DIM, MATMUL_DIM, device=0, dtype=dtype)

    for _ in range(WARMUP_ITERS):
        a @ b
    torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(MATMUL_ITERS):
        a @ b
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    flops = 2 * MATMUL_DIM**3 * MATMUL_ITERS   # one multiply + one add per output element
    return flops / elapsed / 1e12


tf32 = matmul_tflops(torch.float32)
tbf16 = matmul_tflops(torch.bfloat16)

print(f"matmul fp32    : {tf32:7.1f} TFLOP/s")
print(f"matmul bf16    : {tbf16:7.1f} TFLOP/s")
print(f"p2p bandwidth  : {p2p:7.1f} GB/s")
print()
print(f"In the time it takes to send 1 GB to a neighbour GPU, this machine can do")
print(f"about {tbf16 * 1000 / p2p:.0f} TFLOP of bf16 math. Keep that exchange rate in mind:")
print(f"a collective is only free if you have that much compute to hide it behind.")

---
## Worth writing down

Three numbers from this notebook come up again in every module:

1. **Peer-to-peer bandwidth.** The budget for tensor parallelism and for ZeRO's
   all-gathers.
2. **Host bandwidth.** What any offload scheme costs you.
3. **bf16 TFLOP/s.** How much compute you have available to hide communication behind.

Next up: what happens when you ignore all three and simply try to train a model that
does not fit.